In [6]:
import pandas as pd
import numpy as np



## Dataset Overview and Hygiene

### Dataset Structure

In [7]:
df = pd.read_parquet("../outputs/pheme_tweets.parquet")
print("Rows: ", df.shape[0])
print("Columns: ", df.shape[1])
df.info()


Rows:  103212
Columns:  99
<class 'pandas.DataFrame'>
RangeIndex: 103212 entries, 0 to 103211
Data columns (total 99 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   contributors                             0 non-null       object 
 1   truncated                                103212 non-null  bool   
 2   text                                     103212 non-null  str    
 3   in_reply_to_status_id                    96393 non-null   float64
 4   id                                       103212 non-null  int64  
 5   favorite_count                           103212 non-null  int64  
 6   source                                   103212 non-null  str    
 7   retweeted                                103212 non-null  bool   
 8   coordinates                              0 non-null       float64
 9   in_reply_to_screen_name                  96405 non-null   str    
 10  id_str          

### Class distribution


In [8]:
df["event"].value_counts()

event
charliehebdo         38268
ferguson             24175
sydneysiege          23996
ottawashooting       12284
germanwings-crash     4489
Name: count, dtype: int64

In [9]:
df["label_name"].value_counts()

label_name
non-rumours    71782
rumours        31430
Name: count, dtype: int64

In [10]:
pd.crosstab(
    df["event"],
    df["label_name"]
)

label_name,non-rumours,rumours
event,,
charliehebdo,30923,7345
ferguson,17696,6479
germanwings-crash,1995,2494
ottawashooting,5848,6436
sydneysiege,15320,8676


### Missing values

In [6]:
missing = ((df.isnull().mean()*100).sort_values(ascending=False))
high_missing = missing[missing >=95]
print("High missing:--------- ")
print(high_missing)


High missing:--------- 
contributors                       100.000000
place                              100.000000
user_profile_location              100.000000
geo                                100.000000
coordinates                        100.000000
place_attributes_street_address     99.997093
metadata_result_type                99.958338
metadata_iso_language_code          99.958338
possibly_sensitive_appealable       99.284967
entities_trends                     97.736697
filter_level                        97.736697
geo_type                            97.620432
geo_coordinates                     97.620432
coordinates_coordinates             97.620432
coordinates_type                    97.620432
place_contained_within              95.805720
place_country_code                  95.765027
place_url                           95.765027
place_country                       95.765027
place_place_type                    95.765027
place_bounding_box_type             95.765027
place_boun

In [7]:
low_missing = missing[missing <= 5]
print("Low missing:--------- ")
print(low_missing)

Low missing:--------- 
user_following                             2.335969
user_notifications                         2.335969
user_entities_description_urls             2.263303
user_follow_request_sent                   2.263303
user_is_translation_enabled                2.263303
                                             ...   
user_profile_text_color                    0.000000
user_verified                              0.000000
user_profile_background_image_url_https    0.000000
user_id                                    0.000000
n_symbols                                  0.000000
Length: 61, dtype: float64


In [8]:
#checking missing values for important columns
important_cols = [
    "text",
    "lang",
    "event",
    "label_name",
    "thread_id",
    "user_followers_count",
    "user_verified",
    "n_hashtags",
    "n_urls",
    "n_mentions"
]

for col in important_cols:
    missing_pct = df[col].isna().mean() * 100

    print(
        f"{col:25} Missing: {missing_pct:6.2f}%   "
        f"Available: {100-missing_pct:6.2f}%"
    )

text                      Missing:   0.00%   Available: 100.00%
lang                      Missing:   0.00%   Available: 100.00%
event                     Missing:   0.00%   Available: 100.00%
label_name                Missing:   0.00%   Available: 100.00%
thread_id                 Missing:   0.00%   Available: 100.00%
user_followers_count      Missing:   0.00%   Available: 100.00%
user_verified             Missing:   0.00%   Available: 100.00%
n_hashtags                Missing:   0.00%   Available: 100.00%
n_urls                    Missing:   0.00%   Available: 100.00%
n_mentions                Missing:   0.00%   Available: 100.00%


### Language Distribution


In [9]:
(df["lang"].value_counts(normalize=True)*100).round(2)

lang
en     92.21
und     4.16
fr      0.93
es      0.49
in      0.24
ar      0.21
de      0.18
nl      0.17
ru      0.13
tl      0.13
tr      0.13
pt      0.12
ja      0.10
it      0.09
ht      0.07
sv      0.06
pl      0.06
et      0.06
da      0.05
el      0.05
sl      0.04
hi      0.04
ro      0.03
no      0.03
sk      0.03
cy      0.02
fi      0.02
ko      0.02
zh      0.02
th      0.02
ur      0.01
vi      0.01
uk      0.01
lt      0.01
bs      0.01
lv      0.01
am      0.01
hu      0.01
ne      0.01
fa      0.00
hr      0.00
bn      0.00
is      0.00
bg      0.00
iw      0.00
sr      0.00
ml      0.00
Name: proportion, dtype: float64

### Event Time Span

In [10]:
df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce"
)

event_span = (
    df.groupby("event")["created_at"]
    .agg(
        earliest="min",
        latest="max"
    )
)

event_span["duration_days"] = (
    event_span["latest"]
    - event_span["earliest"]
).dt.days

event_span

/var/folders/k3/4xltyhr53vdgsxsvpd_vkhkw0000gn/T/ipykernel_24138/2534404512.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(


,earliest,latest,duration_days
event,,,
charliehebdo,2015-01-07 11:06:08+00:00,2015-01-22 19:23:53+00:00,15
ferguson,2014-08-09 22:33:06+00:00,2014-09-14 16:21:57+00:00,35
germanwings-crash,2015-03-24 10:37:41+00:00,2015-04-01 20:12:15+00:00,8
ottawashooting,2014-10-22 13:55:50+00:00,2014-10-26 01:52:24+00:00,3
sydneysiege,2014-12-14 23:02:38+00:00,2014-12-18 22:07:45+00:00,3


### Event x Label confounding 

In [11]:
event_label_pct = pd.crosstab(
    df["event"],
    df["label_name"],
    normalize="index"
) * 100

event_label_pct.round(2)

label_name,non-rumours,rumours
event,,
charliehebdo,80.81,19.19
ferguson,73.20,26.80
germanwings-crash,44.44,55.56
ottawashooting,47.61,52.39
sydneysiege,63.84,36.16


### Duplicate tweet check

In [12]:
dup_pct = (
    df["text"]
    .duplicated()
    .mean()
    * 100
)

print(f"{dup_pct:.2f}%")

dups = df[
    df["text"].duplicated(keep=False)
]

dups[["id","text"]].head(20)

1.37%


,id,text
52,524963166821048320,@cnni
396,524925553070329856,@CBCNews
444,524967907298910209,@FoxNews
466,524960156669726720,@cnnbrk انت مثقف ؟ حاصل على دكتوراه ؟ تعال هنا...
477,524960144858554368,@cnnbrk ههههه شوفوآ علا الفارس مين متابعه ؟ @Q...
533,524987566970658817,@cnnbrk ههههه شوفوآ علا الفارس مين متابعه ؟ @Q...
534,525012388228382720,@CujoTheG @cnnbrk Fuck you
540,524987571785707520,@cnnbrk انت مثقف ؟ حاصل على دكتوراه ؟ تعال هنا...
544,524988338810683393,@CujoTheG @cnnbrk Fuck you
595,525073999286763522,@globeandmail RIP


### Author overlap analysis


In [13]:
#User apperaing in multiple threads
user_threads = (
    df.groupby("user_id")["thread_id"]
    .nunique()
)

user_threads.describe()

count    49345.000000
mean         1.368102
std          1.887168
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        104.000000
Name: thread_id, dtype: float64

In [14]:
#users active in many threads
int(user_threads[user_threads > 1].count())

8218

In [15]:
#Top Users
user_threads.sort_values(
    ascending=False
).head(20)

user_id
14090948      104
404460410      97
5402612        94
64643056       90
7587032        89
16973333       78
3108351        62
428333         60
292777349      57
1348798826     55
18142983       54
51241574       54
28785486       53
999907196      49
2886586060     49
2097571        48
87818409       48
1367531        48
19038934       44
55794552       43
Name: thread_id, dtype: int64

In [16]:
#users appearing across multiple events
user_events = (
    df.groupby("user_id")["event"]
    .nunique()
)
user_events.describe()

count    49345.000000
mean         1.064687
std          0.287392
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          5.000000
Name: event, dtype: float64

In [17]:
int(user_events[user_events > 1].count())

2752

In [18]:
#Top users
user_events.sort_values(
    ascending=False
).head(20)

user_id
3108351      5
759251       5
15754281     5
972651       5
2097571      5
64643056     5
55794552     5
1367531      5
20639175     5
28785486     5
14173315     5
742143       5
55795410     5
128917051    4
5854302      4
6017542      4
15922161     4
18510860     4
15382667     4
17469289     4
Name: event, dtype: int64